# RAG workshop: how an AI answers questions from your documents

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hugobb/IndigenousAI/blob/main/notebooks/rag_workshop_beginner.ipynb)

An AI chatbot answers from what it learned during its training. It has never read your documents.

RAG (Retrieval-Augmented Generation) fixes this in two steps:
1. It searches your documents for the passages that match the question.
2. It gives those passages to the AI, and the AI writes the answer from them.

In this notebook you try each part of RAG yourself. You don't need to read or write code.

### How this notebook works

- The grey boxes with a ▶️ button are the things you run. To run one, click its ▶️ button (on the left of the box).
- After a few seconds, buttons, sliders or text boxes appear under it. Use them.
- Go from top to bottom.
- If you are stuck, raise your hand.

### The plan

| Part | Topic | Time |
|---|---|---|
| 0 | Setup | 10 min |
| 1 | The problem | 5 min |
| 2 | Embeddings: how a computer compares meanings | 20 min |
| 3 | Search: finding the right passage | 10 min |
| 4 | Chunking: cutting a document into pieces | 20 min |
| 5 | The AI writes the answer | 15 min |
| 6 | Summary and discussion | 10 min |
| Bonus | Your own documents | at home |

## 0. Setup (10 min)

**Do this:**
1. In the menu at the top of the page, click **Runtime**, then **Change runtime type**. Check that **T4 GPU** is selected and click **Save**. (A GPU is a chip that makes the AI fast.)
2. Click ▶️ on the **Setup** box below.
3. If a message says "This notebook was not authored by Google", click **Run anyway**.
4. Wait until the box shows **✅ Ready**. This takes 1 to 2 minutes.

If the box shows ⚠️ instead, everything still works, but the AI is slower. Tell an assistant.

In [ ]:
#@title Setup: click ▶️ and wait for ✅ Ready (1 to 2 minutes) { display-mode: "form" }
# You don't need to read this cell. It installs what is missing, defines everything the
# workshop does, and loads the two AI models. Curious? All the code is below.
try:
    import sentence_transformers, matplotlib, ipywidgets  # noqa: F401
except ImportError:
    %pip install -q sentence-transformers transformers accelerate ipywidgets matplotlib

import re
from dataclasses import dataclass, field
import os
from collections.abc import Iterator
from dataclasses import dataclass
from queue import Empty
from threading import Thread
from typing import Literal
import numpy as np
import html
import io
import time
import ipywidgets as widgets
from matplotlib.figure import Figure
from IPython.display import display


# ======================================================================
# content.py
# ======================================================================
"""Every piece of text the workshop uses: the two documents, the example sentences and the preset questions.

The two documents come from the RAG trust tests kit (corpus/easy/):
- CARE paper: Carroll et al. (2020), The CARE Principles for Indigenous Data Governance,
  Data Science Journal 19:43, https://doi.org/10.5334/dsj-2020-043, CC BY 4.0.
- Example Community Data Charter: FICTIONAL, written for the kit, CC0 1.0.
"""



def _load(name: str) -> str:
    """Read a file from documents/. When the notebook is built, each call is replaced by the text itself."""
    from pathlib import Path

    return (Path(__file__).resolve().parent / "documents" / name).read_text(encoding="utf-8")


@dataclass
class QuestionSpec:
    question: str
    keywords: list[str] = field(default_factory=list)  # a passage "contains the answer" if it has all of them


# ---------------------------------------------------------------- the two documents
def _charter_text(raw: str) -> str:
    """Keep the two-line FICTIONAL label, drop the kit's notes about how the file was made."""
    lines = raw.splitlines()
    return "\n".join(lines[:2] + lines[7:]).strip()


def _care_text(raw: str) -> str:
    """Drop the licence header (shown in the notebook instead) and the '## ' heading marks."""
    body = raw.split("Abstract and main text follow.", 1)[1]
    return re.sub(r"^## (.+)$", r"\1.", body, flags=re.MULTILINE).strip()


CHARTER_TITLE = "Example Community Data Charter (fictional)"
CHARTER = _charter_text('FICTIONAL DOCUMENT - FOR TESTING ONLY.\nThis charter is invented. It does not describe any real community, people, nation, organization or place.\nIt was written for the "RAG trust tests" workshop kit to check whether a question-answering system\nfollows the text it is given, even when that text disagrees with other documents or with what the model\n"already believes". Several statements below deliberately CONTRADICT the published CARE Principles paper\n(Carroll et al. 2020). Do not quote this file as a real policy.\nLicence: CC0 1.0 (public domain dedication) by the kit authors.\n\nExample Community Data Charter (FICTIONAL)\n\nSection 1. Purpose.\nThe Example Community Data Charter sets the rules for how the fictional Example Community Data Council\nlooks after records, recordings and measurements that members share with it.\n\nSection 2. What CARE means in this charter.\nIn this charter, CARE stands for Consent, Access, Review and Exit.\nConsent means that no record is collected without a signed consent form.\nAccess means that members can see every record that is about them.\nReview means that the Council reviews every outside request in public.\nExit means that any member can withdraw a record at any time.\n\nSection 3. Where the principles came from.\nIn this charter, the CARE principles were first written in 2016 at the founding meeting of the\nExample Community Data Council, held in the Council\'s own meeting hall.\n\nSection 4. What FAIR means in this charter.\nIn this charter, FAIR stands for Fast, Affordable, Independent and Reliable.\nThese words describe the computer systems the Council uses, not the data itself.\n\nSection 5. Being CARE Full.\nUnder this charter, a project counts as CARE Full if it meets any two of the four principles.\n\nSection 6. Outside requests.\nOutside researchers must send a written request to the Council.\nThe Council answers every outside request within 45 days.\nA request is approved only if at least seven of the nine Council members vote yes.\n\nSection 7. Withdrawing a record.\nWhen a member asks to withdraw a record, the Council deletes it, including all backup copies,\nwithin 30 days of the request.\n\nSection 8. Storage.\nAll records are kept on two computers in the Council office. No records are stored with outside companies.\n\nSection 9. Review of this charter.\nThe Council reviews this charter every three years.\n')
CARE_TITLE = "The CARE Principles for Indigenous Data Governance (Carroll et al. 2020)"
CARE_PAPER = _care_text('The CARE Principles for Indigenous Data Governance\nStephanie Russo Carroll, Ibrahim Garba, Oscar L. Figueroa-Rodriguez, Jarita Holbrook, Raymond Lovett,\nSimeon Materechera, Mark Parsons, Kay Raseroka, Desi Rodriguez-Lonebear, Robyn Rowe, Rodrigo Sara,\nJennifer D. Walker, Jane Anderson and Maui Hudson.\nData Science Journal, 19: 43, pp. 1-12 (2020). DOI: https://doi.org/10.5334/dsj-2020-043\n\nLicence: "Copyright: (c) 2020 The Author(s). This is an open-access article distributed under the terms of the\nCreative Commons Attribution 4.0 International License (CC-BY 4.0), which permits unrestricted use,\ndistribution, and reproduction in any medium, provided the original author and source are credited.\nSee http://creativecommons.org/licenses/by/4.0/." (from the article PDF, page 12)\n\nChanges made for this workshop kit: converted from the publisher PDF to plain text; page headers,\nline-break hyphens, Figures 1-3 (images; Figure 2 lists the sub-principles, which are therefore not in this text), acknowledgements, competing interests, author contributions\nand the reference list were removed. The wording of the main text is otherwise unchanged.\nAbstract and main text follow.\n\nConcerns about secondary use of data and limited opportunities for benefit-sharing have focused attention on the tension that Indigenous communities feel between (1) protecting Indigenous rights and interests in Indigenous data (including traditional knowledges) and (2) supporting open data, machine learning, broad data sharing, and big data initiatives. The International Indigenous Data Sovereignty Interest Group (within the Research Data Alliance) is a network of nation-state based Indigenous data sovereignty networks and individuals that developed the ‘CARE Principles for Indigenous Data Governance’ (Collective Benefit, Authority to Control, Responsibility, and Ethics) in consultation with Indigenous Peoples, scholars, non-profit organizations, and governments. The CARE Principles are people– and purpose-oriented, reflecting the crucial role of data in advancing innovation, governance, and self-determination among Indigenous Peoples. The Principles complement the existing data-centric approach represented in the ‘FAIR Guiding Principles for scientific data management and stewardship’ (Findable, Accessible, Interoperable, Reusable). The CARE Principles build upon earlier work by the Te Mana Raraunga Maori Data Sovereignty Network, US Indigenous Data Sovereignty Network, Maiam nayri Wingara Aboriginal and Torres Strait Islander Data Sovereignty Collective, and numerous Indigenous Peoples, nations, and communities. The goal is that stewards and other users of Indigenous data will ‘Be FAIR and CARE.’ In this first formal publication of the CARE Principles, we articulate their rationale, describe their relation to the FAIR Principles, and present examples of their application.\n\nKeywords: Indigenous; data sovereignty; data governance; data principles; FAIR principles\n\n## Introduction\n\nAs the world engages with open data, big data, data reuse, and open science, data have increasingly become a global resource used for wielding power, making decisions, spurring innovation and discovery, and commercialization (Independent Expert Advisory Group 2014, The Economist 2017). The effects of this shift are amplified by the adoption of artificial intelligence technologies and the increasing convergence of biological and digital worlds facilitated through data sharing platforms. Historically plagued by data inequities and data exploitation, Indigenous Peoples have raised concerns about the need to integrate Indigenous knowledges and approaches into data practices and policies as both the volume and opportunities for secondary use of data increase (Jackson et al. 2019; Kukutai & Cormack 2019; Carroll et al. 2019; Garrison et al. 2019; Rainie et al. 2019; Kukutai & Taylor 2016b). The articulation of Indigenous Peoples’ rights and interests in data about their peoples, communities, cultures, and territories is part of reclaiming control of data, data ecosystems, data science, and data narratives in the context of open data and open science. Control, coupled with a focus on collective benefit and equity, repositions Indigenous Peoples, nations, and communities from being subjects of data that perpetuate unequal power distributions to self-determining users of data for development and wellbeing. Harnessing data for governance and acting in the governance of data, shifts Indigenous Peoples from invisibility within data ecosystems to vibrant contributors to data policies, practices, ethics, and innovation.\n\n## Leveraging Indigenous Rights for Indigenous Data Futures\n\n## Background\n\nGlobal Indigenous presence spans over 90 countries and includes over 370 million Indigenous persons living out more than 5,000 distinct cultures (UN 2009). Indigenous Peoples have continuity with their pre-colonial societies and seek to use their own social, political, and economic systems to preserve, develop, and transmit to future generations their cultures, knowledges, and relationships with their territories and resources (Martinez Cobo 1982). While often limited by settler-colonial oppression and definition, Indigenous Peoples retain their rights to self-determine their forms of collective governance and rules of membership. In the contemporary world, Indigenous Peoples and nations, people, and communities have become actors across global societies. Indigenous Peoples and their nations are the rightsholders as affirmed by the United Nations and nation-state legal recognition of Indigenous sovereignty (Echo-Hawk 2013; UN 2007; Anaya 2004). Indigenous Peoples as sovereign political collectives often participate in supra- and inter-tribal organizations such as the Oceti Sakowin Oyate and the Iwi Chairs’ Forum, and have local or regional sub-systems such as Tohono O’Odham districts in the US and the Federation of Victorian Traditional Owner Corporations in Australia. Indigenous people often live as citizens of nations (Indigenous) within nations (the nation state, e.g., the United States). Indigenous people also participate in other Indigenous stakeholder groups that do not hold inherent sovereignty, including daily life within Indigenous communities (groups of individuals) within their nations or mainstream society and interaction with Indigenous organizations such as health care entities, non-profits, and businesses.\n\nOngoing processes of colonization of Indigenous Peoples and globalization of Western ideas, values, and lifestyles have resulted in epistemicide, the suppression and co-optation of Indigenous knowledges and data systems (Carroll, Rodriguez-Lonebear & Martinez 2019; Kukutai & Taylor 2016a; Kukutai & Walter 2015; Smith 2012). These processes have limited the ability of Indigenous Peoples to recover, develop, and sustain their knowledges, an ability that is central to Indigenous Peoples’ capacity to realize their human rights and fulfil their responsibilities (Wilson, 2004; Ratima 2001).\n\nAlongside Indigenous aspirations for self-determination, social justice movements have promoted the importance of equity within society and become more deliberate about addressing the negative effects of unconscious bias and structural racism. The pervasiveness of bias across data ecosystems (digital infrastructures, analytics, and applications), within sub-systems (regions/platforms), as well as institutions and communities indicate the importance of addressing inequities at multiple levels. Indigenous people recognize and already incorporate this multi-scale complexity in their knowledge systems and activities; however, this may not be the case for other communities of practice, or knowledge systems.\n\n## The Emergence of Indigenous Data Sovereignty\n\nSince the 1970s, there has been a resurgence in discourse around Indigenous knowledges, identities, and rights, culminating in the 2007 United Nations Declaration on the Rights of Indigenous Peoples [UNDRIP] (UN 2007). UNDRIP reaffirms Indigenous Peoples’ rights to self-determination as political entities and honors the principle of Indigenous control over Indigenous data (Tsosie 2019; Davis 2016). The rights articulated in UNDRIP (especially Article 31) also reflect discourse around Indigenous Cultural and Intellectual Property Rights (ICIP) and Indigenous research ethics (Mātaatua Declaration 1993; Julayinbul Statement 1993; Janke 1998, 2004; Drugge 2016; Tsosie 1997, 2019; Anderson 2009, 2015). UNDRIP reflects a broad approach to Indigenous data that is not restricted by mainstream conceptions of knowledge and intellectual property (Posey & Dutfield, 1996).\n\nIndigenous Peoples’ data include data generated by Indigenous Peoples, as well as by governments and other institutions, on and about Indigenous Peoples and territories. As well as information about Indigenous communities and the individuals, Indigenous and non-Indigenous, that live within (Carroll, Rodriguez-Lonebear & Martinez 2019; Rainie et al. 2019; MnW 2018; Nickerson 2017; TMR 2016). Indigenous Peoples’ data comprise (1) information and knowledge about the environment, lands, skies, resources, and non-humans with which they have relations; (2) information about Indigenous persons such as administrative, census, health, social, commercial, and corporate and, (3) information and knowledge about Indigenous Peoples as collectives, including traditional and cultural information, oral histories, ancestral and clan knowledge, cultural sites, and stories, belongings.\n\nThe advancement of Indigenous self-determination and the reclamation of Indigenous identity and knowledges over the past 50 years has led to the emergence of Indigenous Data Sovereignty, an assertion of the rights and interests of Indigenous Peoples in relation to data about them, their territories, and their ways of life (Carroll, Rodriguez-Lonebear & Martinez 2019; Rainie et al. 2017; Kukutai & Taylor 2016a; Snipp 2016). Indigenous Data Sovereignty has become an increasingly relevant topic as big data, open data, open science, and data reuse become an integral part of research and institutional practices (Carroll, Rodriguez-Lonebear & Martinez 2019; Lovett et al. 2019; Rainie et al. 2019; Kukutai & Taylor 2016a; Taylor & Kukutai 2015). Given that most Indigenous data are held by non-Indigenous governments, institutions, and agencies, increasing Indigenous Peoples’ participation in data governance activities is central to realizing Indigenous Data Sovereignty (Carroll, Rodriguez-Lonebear & Martinez 2019; Rainie et al. 2019; Kukutai & Taylor 2016b; Rodriguez-Lonebear 2016; Smith 2016; Snipp 2016). Indigenous data governance includes both the stewardship and the processes necessary to implement Indigenous control over Indigenous data (collection, storage, analysis, use, reuse) (Carroll, Rodriguez-Lonebear & Martinez 2019; Rainie et al. 2019; Walter 2018; Rodriguez-Lonebear 2016; Smith 2016). The development of conceptual frameworks to inform processes for stewardship and control of Indigenous data has been ongoing in areas such as science, research methods, and research governance (Cajete, 2000; Wilson, 2008; Kovach, 2009; McGregor, Restoule & Johnston, 2018). There is a need to apply insights generated from these frameworks within data ecosystems to help address the challenge of producing data that reflect Indigenous Peoples’ interests and governance needs.\n\n## The Care Principles\n\n## Rationale\n\nMainstream values related to research and data are often inconsistent with Indigenous cultures and collective rights. For instance, data historically belonged to the researcher in mainstream settings, a notion that has shifted under open data and open science movements where publicly funded research requires data sharing. Meanwhile, Indigenous worldviews have typically centered ‘people’ and ‘purpose’ through governance processes that emphasize collective ownership and control of data (Carroll, Rodriguez-Lonebear & Martinez 2019; Kukutai & Taylor 2016b; Smith 2016; Snipp 2016). Indigenous Peoples have been working on ways to authenticate Indigenous forms of knowledge and to fortify their rights to govern research and resulting data (Garrison et al. 2019; Hudson et al. 2016; Smith 2012).\n\nDrawing on the First Nations principles of OCAP® that created data standards for Ownership, Control, Access, and Possession in Canada in the 1990s (FNIGC 2019), Indigenous Data Sovereignty networks in Aotearoa New Zealand, Australia, and the United States, as well as Indigenous scholars, leaders, and allies, determined that there was an urgent need to develop global principles for the governance of Indigenous data. Underlying this need were the primary goals of (1) fostering Indigenous self-determination by enhancing Indigenous use of data for Indigenous pursuits and (2) honoring the ‘FAIR Guiding Principles for scientific data management and stewardship’ (Findable, Accessible, Interoperable, Reusable) (Wilkinson et al. 2016), while ensuring data sharing on Indigenous terms.\n\nThe ‘CARE Principles for Indigenous Data Governance’ (Collective Benefit, Authority to Control, Responsibility, and Ethics) (RDA IG 2019) empower Indigenous Peoples by shifting the focus from regulated consultation to value-based relationships that position data approaches within Indigenous cultures and knowledge systems to the benefit of Indigenous Peoples (Castellano 2004; Anderson et al. 2003). This shift ultimately promotes equitable participation in processes of data reuse, which will result in more equitable outcomes.\n\n## Development\n\nAcademics and practitioners in the International Indigenous Data Sovereignty Interest Group within the Research Data Alliance (RDA 2017) acknowledged the tension between protecting Indigenous rights and interests in data and supporting open data. With this challenge in mind, a workshop was held in Gaborone, Botswana, on Indigenous Data Sovereignty as part of International Data Week in 2018. This Indigenous-led workshop convened Indigenous and allied academics and practitioners to draft principles for Indigenous data governance. Workshop organizers prepared an analysis of principles from frameworks issued by Indigenous Data Sovereignty and governance networks, as well as by mainstream sources, to inform dialogue (Table 1).\n\nParticipants concluded that existing frameworks were oriented toward data, people, and purpose (see Figure 1). While both Indigenous and mainstream principles identified data-centric principles (such as those named in the FAIR Principles), the Indigenous frameworks emphasized people- and purpose-oriented principles.\n\nTable 1: Key Documents Contributing to the Development of the CARE Principles.\n\n- Principles of Māori Data Sovereignty (TMR 2018)\n\n- Indigenous Data Sovereignty Communique (MnW 2018)\n\n- United States Indigenous Data Sovereignty Network Indigenous Data Governance Principles (USIDSN 2020)\n\n- OCAP®: Ownership, Control, Access, and Possession (FNIGC 2018)\n\n- Principles of Ethical Métis Research (MC 2011)\n\n- National Inuit Strategy on Research (ITK 2018)\n\n- Open Data Charter Principles (ODC 2015)\n\n- FAIR Guiding Principles for scientific data management and stewardship (Wilkinson et al. 2016)\n\n- STREAM properties for industrial and commoditized data (Demchenko et al. 2018)\n\n[Figure 1 omitted: Indigenous and Mainstream Data Principles, and Orientation toward Data, People, and Purpose.]\n\nWorkshop participants felt that the mainstream frameworks adequately covered data-centric principles and focused their attention on the ‘people’ and ‘purpose’ foci of the Indigenous principles to create the CARE Principles. Once the four primary CARE Principles of Collective benefit, Authority to control, Responsibility, and Ethics were identified, participants broke into small groups to outline and draft the language for both primary and sub-principles. The resulting draft was shared among nation-state based network members, Indigenous leaders, Indigenous scholars, and allies for input (edits, revisions, comments, feedback) via electronic media, presentations, and engagements.\n\nThe CARE Principles are designed to complement the FAIR Principles and guide the inclusion of Indigenous Peoples in data processes that strengthen Indigenous control for improved discovery, access, use, reuse, and attribution in contemporary data landscapes.\n\n## Exposition\n\nThe CARE Principles address important considerations for modern data ecosystems and across data lifecycles that support both innovation and Indigenous self-determination (Figure 2). The Principles describe high-level actions applicable within research, government, and institutional data settings. The goal is for data stewards and other users of Indigenous data to implement CARE and FAIR Principles in tandem. While each is conceptually distinct, the CARE Principles are related to each other. The Principles define rights, interests, and concepts to be employed in facilitating Indigenous control in data governance and reuse. An enacted mechanism can address a single Principle or multiple simultaneously. However, data governance must address the full set of Principles to demonstrate a CARE Full process.\n\n## The CARE Principles and Their Supporting Concepts\n\nIndigenous data must facilitate collective benefit for Indigenous Peoples to achieve inclusive development and innovation, improve governance and citizen engagement, and realize equitable outcomes. Benefits accrue when data ecosystems are designed and function to support (1) Indigenous nation and community use and reuse of data; (2) use of data for policy decisions and evaluation of services; and (3) creation and use of data that reflect community values.\n\n[Figure 2 omitted: The CARE Principles for Indigenous Data Governance.]\n\nUNDRIP affirms Indigenous Peoples’ rights and interests in their data. Recognition of these rights bolsters Indigenous Peoples’ authority to control and govern such data, further affirming the need for ‘data for governance.’ Indigenous Peoples must have access to data that support Indigenous governance and self-determination. Indigenous Peoples must be the ones to determine data governance protocols, while being actively involved in stewardship decisions for Indigenous data that are held by other entities. When working with Indigenous data, there is a responsibility to nurture respectful relationships with Indigenous Peoples from whom the data originate. Aspects of the relationship include investing in capacity development, increasing community data capabilities, and embedding data within Indigenous languages and cultures. Pursuing these goals fulfills the ultimate responsibility of supporting Indigenous data that advances Indigenous Peoples’ self-determination and collective benefit.\n\nIndigenous Peoples’ rights and wellbeing should be the focus across data ecosystems and throughout data lifecycles in order to minimize harm, maximize benefits, promote justice, and allow for future use. Paramount to ethics in data practices is representation and participation of Indigenous Peoples, who must be the ones to assess benefits, harms, and potential future uses based on community values and ethics.\n\n## The CARE Principles in Action\n\nThe usefulness of high-level Indigenous data governance principles relies on both Indigenous communities and research/data communities to understand operative concepts and to apply them preemptively across data ecosystems and lifecycles. Creating opportunities for non-Indigenous understanding and for Indigenous leadership in data processes will contribute to the development of data and data systems that can lead to Indigenous innovation and development.\n\n## Early Adopters of the CARE Principles\n\nDefining principles is an important step toward exercising Indigenous Data Sovereignty and addressing longstanding concerns about data governance. Some organizations are adopting the CARE Principles through relationships, policies, and mechanisms. Examples include the Research Data Alliance, the Smithsonian Institution, and the Open Data Charter.\n\n## Research Data Alliance (www.rd-alliance.org), in research environments\n\nIn 2016, the Research Data Alliance (RDA) hosted a workshop on Indigenous Data Sovereignty. Recognizing the importance of enhancing Indigenous rights and interests in data and potential implications for mainstream data policy, the RDA supported the creation of the International Indigenous Data Sovereignty Interest Group (IG). The IG convenes existing Indigenous Data Sovereignty networks with other Indigenous and allied researchers, which has resulted in broader global connections, a deeper focus on research data, and the CARE Principles (Lovett et al. 2019; Rainie et al. 2019; RDA IG 2019; RDA 2017). In further support of the IG and the CARE Principles, the RDA formally affiliated with the Global Indigenous Data Alliance (GIDA), which hosts the Principles. The affiliation affirms the importance of both CARE and GIDA, and represents an effort to promote implementation of the Principles throughout the RDA community (RDA 2019).\n\n## The Smithsonian Institution (www.si.edu), in collections environments\n\nIn 2012, the American Museum of Natural History at the Smithsonian Institution in Washington, DC released an Access and Benefit Sharing Policy on Genetic Resources in response to the Nagoya Protocol (2010) addressing Indigenous expectations around benefit sharing, prior informed consent, and mutually agreed terms for collections that it stewards (UN 2011; SI 2012). Subsequently, the US government released the new Open Access to Research Policy in 2013 as an important step in providing unprecedented access to research and data (Stebbins 2013). However, this policy (and others like it) treats all data in the same way: without attention to the nature, the content, or the conditions under which the data was created in the first place (Creative Commons 2019, Wikimedia 2019).\n\nIn December 2019, Smithsonian Directive 609 was released (SI 2019). With specific focus on digital asset access and use, it sets forth Smithsonian policy for access and use of Smithsonian digital assets ahead of the expected release of the institutional Open Access Statement, in February 2020. The SD 609 is important, because it helps identify specific categories of material that might need more attention and care in their management. One of these includes sensitive content, which is: defined in different ways by members of individual communities, nations, tribes, ethnic groups, and religious denominations, but may include materials that relate to traditional knowledge and practices. Such materials may: a) be considered the private domain of specific individuals, clans, cults or societies; b) require an appropriate level of knowledge to view and understand; c) threaten the privacy and well-being of a community when exposed or disclosed to outsiders; and/or d) give offense if inappropriately used or displayed, or when appropriated or exploited for commercial purposes. SD 609 Restriction Code: B1. (SI 2019)\n\nThis identification offers an important opportunity to further explore the incorporation of the CARE Principles into the Smithsonian’s imminent Open Access policy. Recognizing complexity within Smithsonian digital assets and accompanying metadata, the CARE Principles would add an important layer of relational and temporal governance for Indigenous collections within an open access logic. Additionally, there are further opportunities for the CARE Principles to be directly extended into collection management practices. For example, work is about to begin to develop a unique set of ‘Collection CARE Notices’ for Indigenous collections at the Smithsonian and then into other US institutional contexts. Extending the current Cultural Institution Notices developed by Local Contexts (LC 2018), these new Notices will reflect inherent relationships of care, responsibility and governance in collection stewardship within the institution. As they are developed, these Notices will integrate the CARE Principles directly, embedding these into the Smithsonian’s cataloguing and digital infrastructure for Indigenous collections. By providing item- and collection-specific information, the Collection CARE Notices will function as a direct mechanism to assist in management and decision-making consistent with CARE Principles.\n\n## Open Data Charter (opendatacharter.net), in government environments\n\nAfter release of the CARE Principles, the Open Data Charter (ODC) Implementation Working Group hosted a conversation on the Principles (Stone and Calderon 2019). Discussion focused on implementation mechanisms as well as resonance among ODC, FAIR, and CARE Principles (Figure 3). While ODC Principles reflect all four FAIR Principles, they directly address only the CARE Principle of ‘Collective benefit.’ The ODC Principle of ‘Open by default’ does not directly map to the FAIR or CARE Principles. During a review of ODC Principles in 2018, open data community members hotly debated ‘Open by default,’ showing a broad range of interpretations ranging from construing ‘Open by default’ as a key tenet of open data to considering the principle as tough to define and operationalize (Stone and Calderon 2019). The CARE Principles can help refine and specify what ‘Open by default’ means by (1) ensuring roles for Indigenous Peoples and other marginalized communities in open data decision-making and (2) using mechanisms to protect access to data as indicated by these communities to minimize harms and maximize benefits.\n\n[Figure 3 omitted: The Overlap Between the CARE and FAIR Principles and the Open Data Charter.]\n\nWhile implementation has commenced in these examples and elsewhere, further research is needed (1) to identify mechanisms that support implementation of the CARE Principles; (2) to create tools, policies, and practices that enact the Principles; (3) to explore the application of the Principles in different contexts, such as research repositories, knowledge co-production, institutional policies and practices, and, retrospectively, in already existing databases and systems; and (4) to operationalize the Principles in tandem with the FAIR Principles.\n\n## Conclusion\n\nBeing CARE Full is a prerequisite for equitable data and data practices. The CARE Principles draw from, integrate, and build on the work of mainstream stakeholders focused on data for reuse (e.g., FAIR Principles) and the efforts of Indigenous-led networks and coalitions focused on Indigenous data governance and research control. While centering data in the FAIR Principles complements other efforts to inform responsibilities for data producers and repositories, the CARE Principles extend that work to actions that align with the ‘people’ and ‘purpose’ for which data exist and are used. As the mainstream community of data stakeholders advances standards and practices to facilitate data reuse and research reproducibility, the CARE Principles enhance that work by reversing historical power imbalances. The CARE Principles address historical inequities by creating value from Indigenous data in ways that are grounded in Indigenous worldviews and by realizing opportunities for Indigenous Peoples within the knowledge economy.\n\nIn addition, increasingly robust management and stewardship frameworks for Indigenous digital resources to the benefit of Indigenous Peoples have broader implications for other marginalized populations, highlighting the importance of local and group control for the quality and reproducibility of research and data (Rainie et al. 2019). Furthermore, the CARE Principles address issues of relevance for many populations (e.g., privacy, future use, reuse, stewardship), to the extent that the Principles provide a normative foundation for setting standards, crafting policy, and developing research agendas in relation to data acquired about populations as collectives. Those populations might be minority groups, distinctive communities, or other collectives wanting to maintain high levels of trust and accountability in the use of data about their communities. The implementation of the CARE Principles in tandem with the FAIR Principles will result in data that reflect the realities of Indigenous Peoples, be useful for Indigenous purposes, and remain under Indigenous control, while promoting knowledge discovery and innovation. The CARE Principles are a guide for data producers, stewards, and publishers to affirm Indigenous rights to self-determination through CARE Full data practices that will ultimately address complex issues related to privacy, future use, and collective interests, and increase the value of data for reuse. Just as the FAIR Principles are a precursor to good data management and stewardship practices, the CARE Principles encourage data users to be FAIR and CARE.\n')
CARE_CREDIT = (
    "Carroll, S. R., et al. (2020). The CARE Principles for Indigenous Data Governance. "
    "Data Science Journal, 19: 43. https://doi.org/10.5334/dsj-2020-043. "
    "CC BY 4.0. Changes: converted to plain text; figures, headers and references removed."
)

# The charter's sections: the passages searched in section 3.
CHARTER_SECTIONS = [" ".join(s.split()) for s in re.findall(r"Section \d+\..*?(?=\nSection \d+\.|\Z)", CHARTER, flags=re.S)]

# ---------------------------------------------------------------- preset questions (chunking section, about the charter)
DAYS_QUESTION = QuestionSpec("How long does the Council take to answer an outside request?", ["within 45 days"])
VOTE_QUESTION = QuestionSpec(
    "How is an outside request approved?", ["seven of the nine"]
)
CARE_LETTERS_QUESTION = QuestionSpec("What does CARE stand for in the charter?", ["Consent, Access, Review and Exit"])
WITHDRAW_QUESTION = QuestionSpec("How quickly is a withdrawn record deleted?", ["within 30 days"])
CARE_FULL_QUESTION = QuestionSpec("When does a project count as CARE Full?", ["any two of the four"])

CHARTER_QUESTIONS = [DAYS_QUESTION, VOTE_QUESTION, CARE_LETTERS_QUESTION, WITHDRAW_QUESTION, CARE_FULL_QUESTION]

# At DEMO_GOOD_SIZE words, a cut falls inside the answer to this one: chunk ends with CUT_BEFORE, next starts with CUT_AFTER.
SPLIT_QUESTION = VOTE_QUESTION
CUT_BEFORE = "A request is approved"
CUT_AFTER = "only if at least seven"

# Chunk sizes (in words) that the exercises refer to. All verified by tests/test_effects.py.
DEMO_GOOD_SIZE = 60   # every charter question but the vote one is found at this size
DEMO_OVERLAP = 15     # at DEMO_GOOD_SIZE, this overlap puts the vote answer back in one chunk
DEMO_TINY_SIZE = 5    # chunks too short: the 45-day answer is not found, with or without overlap

# Section 1 (the AI alone) and section 5 (the full system, over both documents).
PROBLEM_QUESTION = QuestionSpec(
    "How long does the Example Community Data Council take to answer an outside request?", ["within 45 days"]
)
# Over both documents, search puts this answer 2nd: Passages = 1 misses it, Passages = 3 finds it (tested).
STORAGE_QUESTION = QuestionSpec("Where does the Council keep its records?", ["two computers"])
# Not in either document: a good test of "I don't know".
OUT_OF_DOC_QUESTION = "What is the phone number of the Example Community Data Council?"
# The two documents disagree on this one: the charter redefines CARE.
CONFLICT_QUESTION = "What does CARE stand for?"

# ---------------------------------------------------------------- search section (over CHARTER_SECTIONS)
SEARCH_EXAMPLE = "Can outside companies store our records?"          # Section 8 comes first (tested)
SEARCH_MISS_EXAMPLE = "Who decides if an outside researcher can use the records?"  # answer is Section 6; it is not first (tested)
SEARCH_NOTHING_EXAMPLE = "What's the best pizza topping?"              # every score is low (tested)

# ---------------------------------------------------------------- embedding exercises
SIMILARITY_PAIRS = [
    (  # same meaning, almost no shared words
        "Communities have the right to control data collected about them.",
        "Indigenous peoples should govern how their information is stored and used.",
    ),
    (  # shared word, different meaning
        "Members can withdraw a record at any time.",
        "She broke the world record in the marathon.",
    ),
    (  # unrelated
        "Communities have the right to control data collected about them.",
        "I love to ride my bike on Sunday mornings.",
    ),
]

MAP_SENTENCES = [
    ("No record is collected without a signed consent form.", "asking permission"),
    ("People must agree before their stories are recorded.", "asking permission"),
    ("Elders are asked for permission before photos are taken.", "asking permission"),
    ("Participants can say no to the study at any time.", "asking permission"),
    ("All records are kept on two computers in the office.", "storing data"),
    ("The files are locked in a cabinet and backed up every night.", "storing data"),
    ("Passwords protect the database from strangers.", "storing data"),
    ("The recordings are saved on a server in the community.", "storing data"),
    ("Profits from the research must return to the community.", "sharing benefits"),
    ("The language app was built to help children learn.", "sharing benefits"),
    ("Researchers share their results with the people who took part.", "sharing benefits"),
    ("Jobs created by the project go to local youth.", "sharing benefits"),
    ("Simmer the soup slowly with onions and garlic.", "cooking"),
    ("Fresh bread is best eaten warm from the oven.", "cooking"),
    ("She added a pinch of salt to the sauce.", "cooking"),
    ("We roasted vegetables for the family dinner.", "cooking"),
]


# ======================================================================
# toolkit.py
# ======================================================================
"""The workshop's machinery: chunking, similarity, search, prompts and the two AI models."""



Strategy = Literal["fixed", "overlap", "sentences", "whole"]


# ---------------------------------------------------------------- chunking
def split_sentences(text: str) -> list[str]:
    """Cut text after each '.', '?' or '!'."""
    # Split after . ? ! followed by a capital letter or digit, but not after a lone capital ("E. coli").
    pattern = r"(?<=[.?!])(?<!\b[A-Z]\.)\s+(?=[A-Z0-9\"“‘(])"
    return [s.strip() for s in re.split(pattern, text) if s.strip()]


def chunk_fixed(text: str, size: int) -> list[str]:
    """Cut text into pieces of `size` words, one after the other."""
    words = text.split()
    return [" ".join(words[i : i + size]) for i in range(0, len(words), size)]


def chunk_overlap(text: str, size: int, overlap: int) -> list[str]:
    """Like chunk_fixed, but each piece repeats the last `overlap` words of the previous one."""
    overlap = max(0, min(overlap, size - 1))
    step = size - overlap
    words = text.split()
    chunks = []
    for i in range(0, len(words), step):
        chunks.append(" ".join(words[i : i + size]))
        if i + size >= len(words):
            break
    return chunks


def chunk_sentences(text: str, n: int) -> list[str]:
    """Group text into pieces of `n` sentences."""
    sentences = split_sentences(text)
    return [" ".join(sentences[i : i + n]) for i in range(0, len(sentences), n)]


def make_chunks(text: str, strategy: Strategy, size: int, overlap: int) -> list[str]:
    """One entry point for every chunking strategy. For 'sentences', size = sentences per chunk."""
    if strategy == "fixed":
        return chunk_fixed(text, size)
    if strategy == "overlap":
        return chunk_overlap(text, size, overlap)
    if strategy == "sentences":
        return chunk_sentences(text, size)
    return [text.strip()]


# ---------------------------------------------------------------- similarity and search
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """How close two embeddings point: 1 = same direction (same meaning), 0 = unrelated."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def similarity_label(score: float) -> str:
    """Plain-language reading of a similarity score."""
    if score >= 0.60:
        return "very similar"
    if score >= 0.40:
        return "somewhat related"
    if score >= 0.20:
        return "loosely related"
    return "unrelated"


def rank(query_vec: np.ndarray, doc_vecs: np.ndarray, k: int) -> list[tuple[int, float]]:
    """The k stored texts closest to the query, best first, as (position, score)."""
    scores = [cosine(query_vec, d) for d in doc_vecs]
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [(i, scores[i]) for i in order[: min(k, len(scores))]]


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


class MiniLMEmbedder:
    """Turns text into embeddings (384 numbers each) with a small, fast model."""

    def __init__(self, device: str):
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(EMBED_MODEL, device=device)

    def embed(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


# ---------------------------------------------------------------- prompt
def build_prompt(question: str, contexts: list[str], allow_unknown: bool) -> str:
    """The exact text sent to the model: instructions, the retrieved passages, the question."""
    lines = ["Answer the question using the context below."]
    if allow_unknown:
        lines.append('If the answer is not in the context, say "I don\'t know based on these documents."')
    lines += ["", "Context:"]
    lines += [f"[{i}] {c}" for i, c in enumerate(contexts, start=1)]
    lines += ["", f"Question: {question}", "Answer:"]
    return "\n".join(lines)


# ---------------------------------------------------------------- map of meaning
def project_2d(vectors: np.ndarray) -> np.ndarray:
    """Squash many-number embeddings down to 2 numbers each (PCA), so they can be drawn on a map."""
    x = np.asarray(vectors, dtype=float)
    x = x - x.mean(axis=0)
    out = np.zeros((len(x), 2))
    if len(x) < 2:
        return out
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    proj = x @ vt[:2].T
    out[:, : proj.shape[1]] = proj
    return out


# ---------------------------------------------------------------- the answer-writing model
CHAT_MODEL_GPU = "Qwen/Qwen3-1.7B"
CHAT_MODEL_CPU = "Qwen/Qwen3-0.6B"
MAX_NEW_TOKENS = 200


def model_dtype(device: str):
    """Full precision on the GPU: half precision (float16) can overflow with Qwen3 on a T4."""
    import torch

    return torch.float32 if device == "cuda" else torch.bfloat16


def stream_worker(target, streamer) -> Iterator[str]:
    """Run `target` (the model writing) in the background and pass on its text as it arrives.
    If it fails, stop waiting and raise instead of hanging forever."""
    failure: list[BaseException] = []

    def work():
        try:
            target()
        except BaseException as error:  # noqa: BLE001 - any failure must reach the participant
            failure.append(error)
            streamer.text_queue.put(streamer.stop_signal)

    worker = Thread(target=work, daemon=True)
    worker.start()
    try:
        yield from streamer
    except Empty:
        raise RuntimeError("The model took too long to answer.") from None
    worker.join(timeout=1)
    if failure:
        raise RuntimeError(f"The model stopped with an error: {failure[0]}") from failure[0]


class QwenGenerator:
    """Writes answers with a small Qwen model. Greedy decoding: the same question gives the same answer."""

    def __init__(self, model_name: str, device: str):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=model_dtype(device)).to(device)
        self.model.eval()

    def generate(self, prompt: str) -> Iterator[str]:
        from transformers import TextIteratorStreamer

        text = self.tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=180)
        settings = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
        )
        yield from stream_worker(lambda: self.model.generate(**settings), streamer)


class FakeGenerator:
    """A stand-in for tests: repeats the first passage instead of thinking."""

    def generate(self, prompt: str) -> Iterator[str]:
        yield "(test answer) "
        first = next((line for line in prompt.splitlines() if line.startswith("[1] ")), prompt)
        for word in first.removeprefix("[1] ").split():
            yield word + " "


@dataclass
class Models:
    embedder: MiniLMEmbedder
    generator: QwenGenerator | FakeGenerator
    device: str
    llm_name: str


def load_models() -> Models:
    """Load both models, on the GPU when there is one."""
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = MiniLMEmbedder(device)
    if os.environ.get("RAG_WORKSHOP_FAKE_LLM") == "1":
        return Models(embedder, FakeGenerator(), device, "fake")
    name = os.environ.get("RAG_WORKSHOP_LLM") or (CHAT_MODEL_GPU if device == "cuda" else CHAT_MODEL_CPU)
    return Models(embedder, QwenGenerator(name, device), device, name)


def status_line(device: str) -> str:
    if device == "cuda":
        return "✅ Ready. You can go on to section 1."
    return (
        "⚠️ Ready, but there is no GPU, so the AI is slower (up to a minute per answer) and makes more mistakes. "
        "To fix it: click Runtime > Change runtime type > T4 GPU > Save, then click ▶️ on this box again."
    )


def answer(models: Models, question: str, contexts: list[str], allow_unknown: bool) -> Iterator[str]:
    """The whole RAG step: build the prompt from the retrieved passages and stream the model's answer."""
    return models.generator.generate(build_prompt(question, contexts, allow_unknown))


# ======================================================================
# ui.py
# ======================================================================
"""The interactive panels, one per workshop section. Each show_*() displays its panel and returns it."""




# Mid-tone backgrounds with dark text set explicitly, so they read on light and dark themes.
CHUNK_COLOURS = ["#cfe3ff", "#ffe0b3"]
OVERLAP_COLOUR = "#c8efc4"
TEXT = "color:#111"
WIDE = widgets.Layout(width="100%")


def _panel(children: list, controls: dict) -> widgets.VBox:
    box = widgets.VBox(children, layout=widgets.Layout(width="100%", max_width="900px"))
    box.controls = controls
    display(box)
    return box


def _note(message: str) -> str:
    return f'<div style="padding:8px 12px;border-radius:6px;background:#fff3bf;{TEXT}">{html.escape(message)}</div>'


def _bar(score: float, label: str = "") -> str:
    pct = max(0, min(100, round(score * 100)))
    return (
        '<div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap">'
        f'<div style="width:160px;height:14px;background:#e3e3e3;border-radius:7px;overflow:hidden">'
        f'<div style="width:{pct}%;height:100%;background:#3b82f6"></div></div>'
        f"<b>{score:.2f}</b> {html.escape(label)}</div>"
    )


def _box(inner: str, background: str = "#f4f4f5") -> str:
    return f'<div style="padding:10px 12px;border-radius:6px;background:{background};{TEXT};white-space:pre-wrap">{inner}</div>'


def _once(button: widgets.Button, handler):
    """Attach `handler` to the button, ignoring clicks that were queued while it was busy."""
    state = {"finished": 0.0}

    def on_click(_):
        if time.monotonic() - state["finished"] < 0.5:
            return
        try:
            handler(_)
        finally:
            state["finished"] = time.monotonic()

    button.on_click(on_click)


def _stream_into(target: widgets.HTML, pieces, button: widgets.Button | None = None, background: str = "#eef6ee") -> None:
    """Write the model's answer into `target` as it arrives. The button stays disabled meanwhile."""
    if button is not None:
        button.disabled = True
    text = ""
    target.value = _box("The AI is writing...", background)
    try:
        for piece in pieces:
            text += piece
            target.value = _box(html.escape(text), background)
    except Exception as error:  # noqa: BLE001 - show a plain message instead of a stack trace
        target.value = _box(
            html.escape(text) + "<br><br>⚠️ Something went wrong. Click the button again. "
            "If it happens again, raise your hand."
            f"<br><small>({html.escape(str(error))})</small>",
            "#fdecec",
        )
        return
    finally:
        if button is not None:
            button.disabled = False
    if not text.strip():
        target.value = _box("(The AI gave an empty answer. Click the button again.)", background)


def collection_chunks() -> list[str]:
    """Both documents (the charter and the CARE paper) cut into chunks of DEMO_GOOD_SIZE words, with overlap."""
    return [chunk for doc in (CHARTER, CARE_PAPER) for chunk in make_chunks(doc, "overlap", DEMO_GOOD_SIZE, DEMO_OVERLAP)]


_COLLECTION: dict = {}


def retrieve(m: Models, question: str, k: int) -> list[str]:
    """The k chunks of the two documents closest to the question."""
    key = id(m.embedder)
    if key not in _COLLECTION:
        chunks = collection_chunks()
        _COLLECTION[key] = (chunks, m.embedder.embed(chunks))
    chunks, vecs = _COLLECTION[key]
    hits = rank(m.embedder.embed([question])[0], vecs, k)
    return [chunks[i] for i, _ in hits]


# ---------------------------------------------------------------- 1. the problem
def show_problem(m: Models) -> widgets.VBox:
    question = widgets.HTML(_box(f"<b>Question:</b> {html.escape(PROBLEM_QUESTION.question)}"))
    ask = widgets.Button(description="Ask the AI", button_style="primary", icon="comment")
    out = widgets.HTML()
    _once(ask, lambda _: _stream_into(out, m.generator.generate(PROBLEM_QUESTION.question), ask))
    return _panel([question, ask, out], {"ask": ask, "answer": out})


# ---------------------------------------------------------------- 2a. similarity
def show_similarity(m: Models) -> widgets.VBox:
    labels = ["Same meaning, different words", "Same word, different meaning", "Unrelated"]
    pair = widgets.Dropdown(options=[(lab, i) for i, lab in enumerate(labels)], value=0, description="Example")
    a = widgets.Textarea(value=SIMILARITY_PAIRS[0][0], description="Sentence A", layout=WIDE)
    b = widgets.Textarea(value=SIMILARITY_PAIRS[0][1], description="Sentence B", layout=WIDE)
    compare = widgets.Button(description="Compare", button_style="primary", icon="balance-scale")
    result = widgets.HTML()

    def on_pair(change):
        a.value, b.value = SIMILARITY_PAIRS[change["new"]]
        on_compare(None)

    def on_compare(_):
        if not a.value.strip() or not b.value.strip():
            result.value = _note("Type a sentence first.")
            return
        va, vb = m.embedder.embed([a.value, b.value])
        score = cosine(va, vb)
        result.value = f"Score {_bar(score, '→ ' + similarity_label(score))}"

    pair.observe(on_pair, names="value")
    compare.on_click(on_compare)
    on_compare(None)
    return _panel([pair, a, b, compare, result], {"pair": pair, "a": a, "b": b, "compare": compare, "result": result})


# ---------------------------------------------------------------- 2b. map of meaning
MAP_COLOURS = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd", "#8c564b"]


def _short(sentence: str, words: int = 4) -> str:
    parts = sentence.split()
    return " ".join(parts[:words]) + (" …" if len(parts) > words else "")


def show_meaning_map(m: Models) -> widgets.VBox:
    base = list(MAP_SENTENCES)
    mine: list[str] = []
    sentence = widgets.Text(placeholder="Type any sentence, then click Add", layout=WIDE)
    add = widgets.Button(description="Add to map", button_style="primary", icon="plus")
    reset = widgets.Button(description="Reset", icon="undo")
    message = widgets.HTML()
    plot = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="900px"))
    state: dict = {}

    def redraw():
        # A plain PNG picture: it shows everywhere (Colab, Jupyter, VS Code), with no JavaScript.
        texts = [s for s, _ in base] + mine
        points = project_2d(m.embedder.embed(texts))
        fig = Figure(figsize=(9, 6), dpi=100)
        ax = fig.add_subplot()
        topics = list(dict.fromkeys(t for _, t in base))
        span = points.max(axis=0) - points.min(axis=0) + 1e-9
        placed: list[tuple[float, float]] = []  # label anchors already to the right of their dot

        def label(i: int, text: str, **style) -> None:
            x, y = (points[i] - points.min(axis=0)) / span
            crowded = any(abs(x - px) < 0.25 and abs(y - py) < 0.05 for px, py in placed)
            if not crowded:
                placed.append((x, y))
            ax.annotate(text, points[i], xytext=(-6 if crowded else 6, 4), textcoords="offset points",
                        ha="right" if crowded else "left", **style)

        for topic, colour in zip(topics, MAP_COLOURS):
            idx = [i for i, (_, t) in enumerate(base) if t == topic]
            ax.scatter(points[idx, 0], points[idx, 1], s=70, color=colour, label=topic)
            for i in idx:
                label(i, _short(base[i][0]), fontsize=8, color=colour)
        n = len(base)
        if mine:
            ax.scatter(points[n:, 0], points[n:, 1], s=220, marker="*", color="#111", label="your sentences")
            for i, s in enumerate(mine):
                label(n + i, _short(s), fontsize=9, fontweight="bold", color="#111")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.margins(0.25, 0.12)
        ax.set_title("Each dot is a sentence. Dots close together have similar meanings.", fontsize=11)
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=5, frameon=False, fontsize=9)
        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        plot.value = buf.getvalue()
        state["points"] = points
        state["mine"] = list(mine)

    def on_add(_):
        if not sentence.value.strip():
            message.value = _note("Type a sentence first.")
            return
        mine.append(sentence.value.strip())
        sentence.value = ""
        message.value = ""
        redraw()

    def on_reset(_):
        mine.clear()
        message.value = ""
        redraw()

    add.on_click(on_add)
    reset.on_click(on_reset)
    redraw()
    box = _panel([widgets.HBox([add, reset]), sentence, message, plot],
                 {"sentence": sentence, "add": add, "reset": reset, "message": message, "plot": plot})
    box.map_state = state
    return box


# ---------------------------------------------------------------- 3. search
def show_search(m: Models) -> widgets.VBox:
    question = widgets.Text(value=SEARCH_EXAMPLE, description="Question", layout=WIDE)
    search = widgets.Button(description="Search", button_style="primary", icon="search")
    results = widgets.HTML()
    passage_vecs = m.embedder.embed(CHARTER_SECTIONS)

    def on_search(_):
        if not question.value.strip():
            results.value = _note("Type a question first.")
            return
        hits = rank(m.embedder.embed([question.value])[0], passage_vecs, len(CHARTER_SECTIONS))
        rows = []
        for place, (i, score) in enumerate(hits, start=1):
            bg = "#dbeafe" if place <= 3 else "#f4f4f5"
            rows.append(
                f'<div class="rag-row" style="padding:8px 10px;margin:4px 0;border-radius:6px;background:{bg};{TEXT}">'
                f"<b>#{place}</b> {_bar(score, '→ ' + similarity_label(score))}"
                f'<div style="margin-top:4px">{html.escape(CHARTER_SECTIONS[i])}</div></div>'
            )
        results.value = "<div>The 3 sections in blue are the ones RAG would give to the AI.</div>" + "".join(rows)

    search.on_click(on_search)
    on_search(None)
    return _panel([question, search, results], {"question": question, "search": search, "results": results})


# ---------------------------------------------------------------- 4. chunking
def _render_chunks(words: list[str], strategy: str, chunks: list[str], size: int, overlap: int, winner: int) -> str:
    """The report with each chunk in alternating colours; words shared by two chunks in green."""
    owners: list[list[int]] = [[] for _ in words]
    if strategy == "overlap":
        step = size - max(0, min(overlap, size - 1))
        for ci in range(len(chunks)):
            for w in range(ci * step, min(ci * step + size, len(words))):
                owners[w].append(ci)
    else:
        pos = 0
        for ci, chunk in enumerate(chunks):
            for w in range(pos, min(pos + len(chunk.split()), len(words))):
                owners[w].append(ci)
            pos += len(chunk.split())
    parts = []
    for word, own in zip(words, owners):
        colour = OVERLAP_COLOUR if len(own) > 1 else CHUNK_COLOURS[own[0] % 2] if own else "transparent"
        style = f"background:{colour};{TEXT}"
        if winner in own:
            style += ";font-weight:bold;text-decoration:underline"
        parts.append(f'<span style="{style}">{html.escape(word)}</span>')
    return (
        f'<div style="max-height:320px;overflow-y:auto;padding:10px;border:1px solid #ccc;border-radius:6px;'
        f'background:#fff;{TEXT};line-height:1.7">' + " ".join(parts) + "</div>"
    )


def show_chunking(m: Models) -> widgets.VBox:
    strategy = widgets.Dropdown(
        options=[("Fixed size", "fixed"), ("Fixed size with overlap", "overlap"),
                 ("By sentences", "sentences"), ("Whole document (no chunking)", "whole")],
        value="fixed", description="How to cut",
    )
    size = widgets.IntSlider(value=DEMO_GOOD_SIZE, min=5, max=200, step=5, description="Words/chunk",
                             continuous_update=False)
    overlap = widgets.IntSlider(value=DEMO_OVERLAP, min=0, max=50, step=5, description="Overlap",
                                disabled=True, continuous_update=False)
    question = widgets.Dropdown(options=[q.question for q in CHARTER_QUESTIONS], description="Question",
                                layout=widgets.Layout(width="100%", max_width="560px"))
    count = widgets.HTML()
    text = widgets.HTML()
    top = widgets.HTML()
    ask = widgets.Button(description="Ask the AI with this chunk", button_style="primary", icon="comment",
                         layout=widgets.Layout(width="auto"))
    out = widgets.HTML()
    words = CHARTER.split()
    cache: dict = {}
    state = {"switching": False, "top": ""}

    def current_chunks():
        key = (strategy.value, size.value, overlap.value if strategy.value == "overlap" else 0)
        if key not in cache:
            chunks = make_chunks(CHARTER, strategy.value, size.value, key[2])
            cache[key] = (chunks, m.embedder.embed(chunks))
        return cache[key]

    def update(_=None):
        if state["switching"]:
            return
        overlap.disabled = strategy.value != "overlap"
        size.disabled = strategy.value == "whole"
        chunks, vecs = current_chunks()
        (i, score), = rank(m.embedder.embed([question.value])[0], vecs, 1)
        state["top"] = chunks[i]
        count.value = f"{CHARTER_TITLE}: cut into <b>{len(chunks)} chunk{'s' if len(chunks) != 1 else ''}</b>"
        text.value = _render_chunks(words, strategy.value, chunks, size.value, overlap.value, i)
        top.value = (
            f"<b>Chunk picked by search</b> (chunk {i + 1}), score {_bar(score)}"
            + _box(html.escape(chunks[i]), "#dbeafe")
        )
        out.value = ""

    def on_strategy(change):
        state["switching"] = True
        if change["new"] == "sentences":
            size.min, size.step, size.value, size.max = 1, 1, 3, 10
            size.description = "Sentences"
        elif change["old"] == "sentences":
            size.max, size.value, size.min, size.step = 200, DEMO_GOOD_SIZE, 5, 5
            size.description = "Words/chunk"
        state["switching"] = False
        update()

    def on_ask(_):
        contexts = [state["top"]]
        _stream_into(out, answer(m, question.value, contexts, allow_unknown=True), ask)

    strategy.observe(on_strategy, names="value")
    for w in (size, overlap, question):
        w.observe(update, names="value")
    _once(ask, on_ask)
    update()
    controls = {"strategy": strategy, "size": size, "overlap": overlap, "question": question,
                "count": count, "text": text, "top": top, "ask": ask, "answer": out}
    return _panel([widgets.HBox([strategy, size, overlap], layout=widgets.Layout(flex_flow="row wrap")),
                   question, count, text, top, ask, out], controls)


# ---------------------------------------------------------------- 5. generation
def show_generation(m: Models) -> widgets.VBox:
    question = widgets.Text(value=PROBLEM_QUESTION.question, description="Question", layout=WIDE)
    topk = widgets.IntSlider(value=3, min=1, max=5, description="Passages")
    unknown = widgets.Checkbox(value=True, description="Let the AI say I don't know", indent=False)
    ask = widgets.Button(description="Ask", button_style="primary", icon="comment")
    prompt = widgets.HTML()
    rag = widgets.HTML()
    norag = widgets.HTML()
    column = widgets.Layout(flex="1 1 300px", min_width="260px", margin="4px")
    columns = widgets.Box(
        [widgets.VBox([widgets.HTML("<b>With RAG</b> (the AI read the passages)"), rag], layout=column),
         widgets.VBox([widgets.HTML("<b>Without RAG</b> (the AI's memory only)"), norag], layout=column)],
        layout=widgets.Layout(display="flex", flex_flow="row wrap", width="100%"),
    )
    controls = {"question": question, "topk": topk, "unknown": unknown, "ask": ask,
                "prompt": prompt, "rag": rag, "norag": norag, "prompt_text": ""}

    def on_ask(_):
        q = question.value.strip()
        if not q:
            prompt.value = _note("Type a question first.")
            return
        ask.disabled = True
        try:
            contexts = retrieve(m, q, topk.value)
            text = build_prompt(q, contexts, unknown.value)
            controls["prompt_text"] = text
            shown = html.escape(text)
            for ctx in contexts:
                esc = html.escape(ctx)
                shown = shown.replace(esc, f'<mark style="background:#dbeafe;{TEXT}">{esc}</mark>', 1)
            prompt.value = "<b>The message sent to the AI:</b>" + _box(shown)
            rag.value = norag.value = ""
            _stream_into(rag, m.generator.generate(text))
            _stream_into(norag, m.generator.generate(q), background="#fdecec")
        finally:
            ask.disabled = False

    _once(ask, on_ask)
    return _panel([question, widgets.HBox([topk, unknown], layout=widgets.Layout(flex_flow="row wrap")),
                   ask, prompt, columns], controls)


# ---------------------------------------------------------------- bonus: your own documents
def load_uploads(files: dict[str, bytes]) -> tuple[list[str], list[str]]:
    """Turn uploaded files into chunks. Returns (chunks, messages about skipped files)."""
    chunks, messages = [], []
    for name, data in files.items():
        if not name.lower().endswith((".txt", ".md")):
            messages.append(f"⚠️ {name} is not a .txt file (skipped). Save it as plain text (.txt) and try again.")
            continue
        try:
            text = bytes(data).decode("utf-8")
        except UnicodeDecodeError:
            messages.append(f"⚠️ {name} could not be read as text (skipped). Save it as UTF-8 plain text and try again.")
            continue
        if not text.strip():
            messages.append(f"⚠️ {name} is empty (skipped).")
            continue
        chunks += make_chunks(text, "overlap", DEMO_GOOD_SIZE, DEMO_OVERLAP)
    return chunks, messages


def files_from_widget(value) -> dict[str, bytes]:
    """FileUpload's value as {name: bytes}, for ipywidgets 8 (tuple of dicts) and 7 (dict, as in Colab)."""
    if isinstance(value, dict):
        return {name: bytes(item["content"]) for name, item in value.items()}
    return {item["name"]: bytes(item["content"]) for item in value}


def show_upload(m: Models) -> widgets.VBox:
    status = widgets.HTML()
    question = widgets.Text(placeholder="Ask a question about your files", description="Question", layout=WIDE)
    ask = widgets.Button(description="Ask", button_style="primary", icon="comment")
    out = widgets.HTML()
    store = {"chunks": [], "vecs": None}

    def take(files: dict[str, bytes]):
        chunks, messages = load_uploads(files)
        store["chunks"] = chunks
        store["vecs"] = m.embedder.embed(chunks) if chunks else None
        lines = [html.escape(msg) for msg in messages]
        lines.append(f"✅ {len(chunks)} chunks ready." if chunks else "No usable text yet.")
        status.value = _box("<br>".join(lines))

    picker = widgets.FileUpload(accept=".txt,.md", multiple=True, description="Choose .txt files")
    picker.observe(lambda change: take(files_from_widget(change["new"])), names="value")

    def on_ask(_):
        if not store["chunks"]:
            out.value = _note("Upload at least one .txt file first.")
            return
        if not question.value.strip():
            out.value = _note("Type a question first.")
            return
        hits = rank(m.embedder.embed([question.value])[0], store["vecs"], 3)
        contexts = [store["chunks"][i] for i, _ in hits]
        _stream_into(out, answer(m, question.value, contexts, allow_unknown=True), ask)

    _once(ask, on_ask)
    return _panel([picker, status, question, ask, out],
                  {"picker": picker, "status": status, "question": question, "ask": ask, "answer": out})


print("Loading the two AI models. This takes 1 to 2 minutes...")
MODELS = load_models()
print(status_line(MODELS.device))


## 1. The problem (5 min)

In this workshop we use two documents:
- **The CARE Principles for Indigenous Data Governance** (Carroll et al. 2020), a real paper.
- **The Example Community Data Charter**, the rules of a community data council. This charter is **made up** for the workshop, so the AI has never seen it.

First, let's ask the AI a question about the charter without giving it the charter.

**Do this:**
1. Click ▶️ on the box below.
2. Click **Ask the AI**.
3. Read the answer. The charter says: the Council answers within 45 days. Did the AI get it right?

The AI does not know the charter, but it usually answers anyway and makes something up. This is called a **hallucination**.

RAG solves this. Before the AI answers, RAG finds the right part of the charter and gives it to the AI. Sections 2 to 5 show how.

In [ ]:
#@title Click ▶️: ask the AI without the charter { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Run the Setup box first (section 0, at the top). Then click ▶️ on this box again.")
else:
    show_problem(MODELS)


## 2. Embeddings: how a computer compares meanings (20 min)

To search by meaning, the computer turns each sentence into a list of 384 numbers. This list is called an **embedding**. Two sentences with similar meanings get similar numbers, even if they use different words.

By comparing two embeddings, the computer gives a **similarity score**: 0 means unrelated, and the higher the score, the closer the meanings. With this model, scores above 0.6 are rare, and 0.4 is already close.

### 2a. Compare two sentences

**Do this:**
1. Click ▶️ on the box below. You see two sentences and their score.
2. In the **Example** menu, choose each example in turn:
   - "Same meaning, different words": a high score, even though the sentences share almost no words.
   - "Same word, different meaning": a low score, even though both sentences contain the word "record".
   - "Unrelated": a low score.
3. Write your own sentences in **Sentence A** and **Sentence B**, then click **Compare**. Try "I love winter." and "I hate winter."

The score tells you if two sentences talk about the same thing, not if they agree. "I love winter." and "I hate winter." get a high score.

In [ ]:
#@title Click ▶️: compare two sentences { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Run the Setup box first (section 0, at the top). Then click ▶️ on this box again.")
else:
    show_similarity(MODELS)


### 2b. The map of meanings

The map shows 16 sentences. Each dot is a sentence. Dots close together have similar meanings. The colours show 4 topics.

**Do this:**
1. Click ▶️ on the box below.
2. Read the first words of each sentence next to its dot.
3. Type "Nobody may record a ceremony without the family's permission." in the text box, then click **Add to map**. Your sentence appears as a black star. Which colour is it closest to?
4. Add your own sentences. Click **Reset** to remove them.

Nobody told the computer what the topics are. It places sentences with similar meanings close together on its own.

In [ ]:
#@title Click ▶️: the map of meanings { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Run the Setup box first (section 0, at the top). Then click ▶️ on this box again.")
else:
    show_meaning_map(MODELS)


<details><summary>More detail (optional)</summary>

The embedding model is `all-MiniLM-L6-v2`. The similarity score is the cosine similarity between the two lists of numbers. To draw the map, the 384 numbers are reduced to 2 with a method called PCA, so the map is only an approximation.

</details>

## 3. Search: finding the right passage (10 min)

Search uses embeddings. The computer keeps an embedding for every passage. When you ask a question, it computes the score between your question and each passage, and sorts the passages from the highest score to the lowest.

Here the passages are the 9 sections of the charter.

**Do this:**
1. Click ▶️ on the box below. It has already searched for "Can outside companies store our records?". Section 8 (Storage) comes first. Correct.
2. Replace the question with "Who decides if an outside researcher can use the records?" and click **Search**. The right answer is Section 6 (Outside requests): the Council votes. Is Section 6 first?
3. Search for "What's the best pizza topping?". No section is about pizza. Look at the scores.

Search is not perfect. It can put the wrong passage first. And it always returns passages, even when none of them is relevant.

In [ ]:
#@title Click ▶️: search the charter { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Run the Setup box first (section 0, at the top). Then click ▶️ on this box again.")
else:
    show_search(MODELS)


## 4. Chunking: cutting a document into pieces (20 min)

Real documents are long. One embedding for a whole document is too vague, so the document is cut into small pieces called **chunks**. Each chunk gets its own embedding, and search returns the best chunk.

In the box below, the charter is cut into chunks. The chunks are coloured blue and orange in turn. Words in green belong to two chunks (this is called **overlap**). The chunk picked by search is in bold and underlined, and is shown again below the text.

**Do this:**
1. Click ▶️ on the box below. Leave **Question** on "How long does the Council take to answer an outside request?" and click **Ask the AI with this chunk**. The AI answers: within 45 days. Correct.
2. In **How to cut**, choose **Whole document (no chunking)**. The score goes down: one embedding for the whole charter is too vague.
3. Choose **Fixed size** again and move **Words/chunk** to **5**. Click **Ask the AI with this chunk**. The chunks are now too short to hold the answer.
4. Move **Words/chunk** back to **60**. In **Question**, choose "How is an outside request approved?". In the text, find "A request is approved": the chunk ends there, in the middle of the answer. Click **Ask the AI with this chunk**.
5. In **How to cut**, choose **Fixed size with overlap**, and set **Overlap** to **15**. Each chunk now repeats the last 15 words of the chunk before it. The answer is in one chunk again. Click **Ask the AI with this chunk**.
6. Try the other questions, and the **By sentences** option.

Chunks that are too big are vague. Chunks that are too small, or cut in the wrong place, lose the answer. Overlap helps.

In [ ]:
#@title Click ▶️: cut the charter into chunks { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Run the Setup box first (section 0, at the top). Then click ▶️ on this box again.")
else:
    show_chunking(MODELS)


## 5. The AI writes the answer (15 min)

This is the full RAG system. It searches both documents (the charter and the CARE paper, cut into chunks of 60 words with an overlap of 15). It takes the best chunks, puts them in a message with the question and a short instruction, and sends the message to the AI.

The box shows the exact message the AI receives. The chunks are in blue.

**Do this:**
1. Click ▶️ on the box below, then click **Ask**. Compare the two answers: on the left the AI read the chunks (with RAG), on the right it only used its memory (without RAG).
2. In **Question**, type "Where does the Council keep its records?". Set **Passages** to **1** and click **Ask**. Then set **Passages** to **3** and click **Ask** again. With 1 passage, the AI does not get the chunk that has the answer (two computers in the Council office).
3. Type "What is the phone number of the Example Community Data Council?" and click **Ask**. Neither document says. With **Let the AI say I don't know** ticked, the AI should say it doesn't know. Untick it and click **Ask** again.
4. Type "What does CARE stand for?" and click **Ask**. The two documents disagree: the CARE paper says Collective Benefit, Authority to Control, Responsibility, Ethics; the made-up charter says Consent, Access, Review, Exit. Which answer does the AI give? Look at which chunks it received.

The AI can only answer as well as the chunks it receives. When search gives it the wrong chunks, the answer is wrong or missing.

In [ ]:
#@title Click ▶️: ask with and without RAG { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Run the Setup box first (section 0, at the top). Then click ▶️ on this box again.")
else:
    show_generation(MODELS)


## 6. Summary and discussion (10 min)

### RAG in short

1. Once, in advance: cut the documents into chunks and compute an embedding for each chunk.
2. For each question: compute the question's embedding, find the closest chunks, send them to the AI with the question, and get the answer.

Each choice changes the answers: chunk size, overlap, the embedding model, the number of passages, and the instruction given to the AI.

### Running it on your own computer

In Colab, your text is processed on Google's computers. On your own computer, your documents stay with you. This matters for sensitive documents.

Two options (they need some help from someone comfortable with computers):
- **This notebook.** Install Python 3.11 or newer. In a terminal, run `pip install sentence-transformers transformers torch accelerate ipywidgets matplotlib jupyterlab`. Download this notebook (**File > Download > Download .ipynb**), then start `jupyter lab` and open it.
- **Ollama**, a free app that runs AI models on your computer. The notebook [`ollama_rag_workshop.ipynb`](https://github.com/hugobb/IndigenousAI/blob/main/notebooks/ollama_rag_workshop.ipynb) builds the same system with Ollama and shows all the code.

### What should go into such a system?

Running the AI on your own computer does not answer a more important question: should these documents go into an AI system at all? That is a decision for the community the documents come from, and it comes before the technology.

The **OCAP® principles** (Ownership, Control, Access, Possession), from the First Nations Information Governance Centre, say that First Nations communities own and control their data. The **CARE Principles** you used today (Collective Benefit, Authority to Control, Responsibility, Ethics) say the same for Indigenous data worldwide.

**To discuss:**
1. Who should decide which documents go into a RAG system about a community, and who may ask it questions?
2. Even with RAG, the AI can be wrong. Who is responsible for a wrong answer?
3. In your own work, what would you put into such a system, and what would you never put in?

### Sources

- Carroll, S. R., et al. (2020). The CARE Principles for Indigenous Data Governance. Data Science Journal, 19: 43. https://doi.org/10.5334/dsj-2020-043. CC BY 4.0. Changes: converted to plain text; figures, headers and references removed.
- Example Community Data Charter: fictional, written for the RAG trust tests kit, CC0 1.0. It describes no real community.

## Bonus: your own documents

You can ask questions about your own text files.

⚠️ **In Colab, uploaded files are sent to Google's computers.** Only upload text you are allowed to share. Don't upload anything sensitive.

**Do this:**
1. Save your document as a plain text file. In Word or Google Docs: **File > Download > Plain text (.txt)**.
2. Click ▶️ on the box below, then click **Choose .txt files** and pick your file.
3. Type a question in **Question** and click **Ask**.

In [ ]:
#@title Click ▶️: your own documents { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Run the Setup box first (section 0, at the top). Then click ▶️ on this box again.")
else:
    show_upload(MODELS)
